# Banking Customer Analytics & Risk Engine — Customer Segmentation

This notebook demonstrates the unsupervised borrower segmentation workflow:
1. Feature matrix construction strictly excluding the target label
2. Standard feature scaling (`StandardScaler`)
3. Candidate cluster evaluation ($k=2..10$) via elbow inertia and silhouette coefficients
4. Final K-Means model training with fixed random seed
5. Cluster stability verification across multiple random seeds (Adjusted Rand Index)
6. Segment profiling, persona labeling, and observed approval rate audits
7. 2D PCA projection and cluster separation inspection
8. Cross-referencing supervised predicted probabilities with unsupervised borrower clusters

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from banking_analytics.utils.config_loader import load_config
from banking_analytics.utils.io_utils import read_csv_safe
from banking_analytics.segmentor import (
    build_clustering_features,
    evaluate_k_range,
    check_clustering_stability,
    build_cluster_profiles,
)
from banking_analytics.classifier import train_and_evaluate_classification
from banking_analytics.evaluator import cross_reference_clusters
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

config = load_config("../config/config.yaml")
df = read_csv_safe(f"../{config['data']['processed_path']}")
print(f"Loaded clean dataset: {df.shape[0]} rows, {df.shape[1]} columns")

## 1. Feature Preprocessing for Unsupervised Segmentation
The target variable (`Loan_Approved`) is strictly excluded.

In [ ]:
X_scaled, feature_names, scaler = build_clustering_features(df)
print(f"Scaled feature matrix shape: {X_scaled.shape}")
print(f"Features included ({len(feature_names)}): {feature_names[:8]}...")

## 2. Evaluate Candidate Cluster Range (k = 2 to 10)

In [ ]:
k_range, inertias, silhouettes = evaluate_k_range(X_scaled, k_min=2, k_max=6)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(k_range, inertias, 'bo-')
ax1.set_title("Inertia (Elbow Method)")
ax1.set_xlabel("k")

ax2.plot(k_range, silhouettes, 'rs-')
ax2.set_title("Silhouette Score")
ax2.set_xlabel("k")
plt.show()

## 3. Fit Final K-Means Model (k = 4) & Check Stability

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)

stability = check_clustering_stability(X_scaled, k=4, seeds=[0, 42, 123, 456, 789])
print(f"Stability Mean ARI across 5 seeds: {stability['mean_ari']:.4f} (Is Stable: {stability['is_stable']})")

## 4. Segment Profiling & Risk Personas

In [ ]:
profiles = build_cluster_profiles(df, labels)
display(profiles[[
    'cluster_id', 'persona_label', 'size', 'pct_of_portfolio',
    'avg_credit_score', 'avg_income', 'avg_dti', 'approval_rate', 'dominant_loan_purpose'
]])

## 5. 2D PCA Cluster Visualization

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=labels, palette='Set2', alpha=0.5)
plt.title("2D PCA Projection of Borrower Segments")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()